# Bronze
Raw ingestion (/Volumes/workspace/default/raw_data)

In [0]:
%sql
-- Creating and selecting the Catalog
CREATE CATALOG IF NOT EXISTS accenture_final_project;
USE CATALOG accenture_final_project;

In [0]:
%sql
-- Creation of the 3 Schemas (Databases)
CREATE SCHEMA IF NOT EXISTS bronze_layer;
CREATE SCHEMA IF NOT EXISTS silver_layer;
CREATE SCHEMA IF NOT EXISTS gold_layer;

# **Targets**

In [0]:
%sql
CREATE OR REPLACE TABLE bronze_layer.Targets
USING DELTA
AS
SELECT * FROM read_files(
  '/Volumes/workspace/default/raw_data/Targets.csv',
  format => 'csv',
  sep => '\t',
  header => true,
  inferSchema => true,
  timestampFormat => 'yyyy-MM-dd HH:mm:ss',
  dateFormat => 'yyyy-MM-dd',
  mode => 'PERMISSIVE'
);

In [0]:
%sql
SELECT * FROM bronze_layer.Targets

### **Validate**

In [0]:
%sql
DESCRIBE HISTORY bronze_layer.Targets;

# **Salesperson**

In [0]:
%sql
CREATE OR REPLACE TABLE bronze_layer.Salesperson
USING DELTA
AS
SELECT * FROM read_files(
  '/Volumes/workspace/default/raw_data/Salesperson.csv',
  format => 'csv',
  sep => '\t',
  header => true,
  inferSchema => true,
  timestampFormat => 'yyyy-MM-dd HH:mm:ss',
  dateFormat => 'yyyy-MM-dd',
  mode => 'PERMISSIVE'
);

In [0]:
%sql
SELECT * FROM bronze_layer.Salesperson

### **Validate**

In [0]:
%sql
DESCRIBE HISTORY bronze_layer.Salesperson;

# **Product**

In [0]:
# 1. Διαβάζουμε το αρχείο σε ένα DataFrame
df = spark.read.format("csv") \
    .option("sep", "\t") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/raw_data/Product.csv")

# 2. Αυτοματοποιημένος καθαρισμός (Αντικατάσταση κενών κλπ)
for col_name in df.columns:
    df = df.withColumnRenamed(col_name, col_name.replace(" ", "_").replace(";", "").replace("(", "").replace(")", ""))

# 3. ΕΠΙΒΟΛΗ ΤΗΣ ΣΩΣΤΗΣ ΣΕΙΡΑΣ
correct_order = [
    "ProductKey",
    "Product",
    "Standard_Cost",
    "Color",
    "Subcategory",
    "Category",
    "Background_Color_Format",
    "Font_Color_Format"
]
df = df.select(*correct_order)

# 4. Αποθήκευση σε Delta Table (με overwriteSchema για να πάρει τη νέα σειρά)
df.write.format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("bronze_layer.Product")

In [0]:
%sql
SELECT * FROM bronze_layer.Product

### **Validate**

In [0]:
%sql
DESCRIBE HISTORY bronze_layer.Salesperson;

# **Region**

In [0]:
%sql
CREATE OR REPLACE TABLE bronze_layer.Region   
USING DELTA
AS
SELECT * FROM read_files(
  '/Volumes/workspace/default/raw_data/Region.csv',
  format => 'csv',
  sep => '\t',
  header => true,
  inferSchema => true,
  timestampFormat => 'yyyy-MM-dd HH:mm:ss',
  dateFormat => 'yyyy-MM-dd',
  mode => 'PERMISSIVE'
);

In [0]:
%sql
SELECT * FROM bronze_layer.Region

### **Validate**

In [0]:
%sql
DESCRIBE HISTORY bronze_layer.Region;

# **Reseller**

In [0]:
# 1. Διαβάζουμε το αρχείο
df = spark.read.format("csv") \
    .option("sep", "\t") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
    .option("dateFormat", "yyyy-MM-dd") \
    .option("mode", "PERMISSIVE") \
    .load("/Volumes/workspace/default/raw_data/Reseller.csv")

# 2. Φτιάχνουμε τα νέα ονόματα (βγάζουμε τα κενά) διατηρώντας την ΊΔΙΑ σειρά
clean_columns = [col.replace(" ", "_") for col in df.columns]

# 3. Εφαρμόζουμε τα νέα ονόματα μαζικά 
df = df.toDF(*clean_columns)

# 4. Αποθηκεύουμε στον πίνακα
df.write.format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("bronze_layer.Reseller")

In [0]:
%sql
SELECT * FROM bronze_layer.Reseller

### **Validate**

In [0]:
%sql
DESCRIBE HISTORY bronze_layer.Reseller;

# **SalespersonRegion**

In [0]:
%sql
CREATE OR REPLACE TABLE bronze_layer.SalespersonRegion   
USING DELTA
AS
SELECT * FROM read_files(
  '/Volumes/workspace/default/raw_data/SalespersonRegion.csv',
  format => 'csv',
  sep => '\t',
  header => true,
  inferSchema => true,
  timestampFormat => 'yyyy-MM-dd HH:mm:ss',
  dateFormat => 'yyyy-MM-dd',
  mode => 'PERMISSIVE'
);

In [0]:
%sql
SELECT * FROM bronze_layer.SalespersonRegion

### **Validate**

In [0]:
%sql
DESCRIBE HISTORY bronze_layer.SalespersonRegion;

# **Sales(2017-2020)**

In [0]:
# 1. Διαβάζουμε το αρχείο
df_sales = spark.read.format("csv") \
    .option("sep", "\t") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
    .option("dateFormat", "yyyy-MM-dd") \
    .load("/Volumes/workspace/default/raw_data/Sales_2017.csv")

# 2. Αυτοματοποιημένος καθαρισμός ονομάτων (Unit Price -> Unit_Price)
for col_name in df_sales.columns:
    df_sales = df_sales.withColumnRenamed(col_name, col_name.replace(" ", "_").replace(";", "").replace("(", "").replace(")", ""))

# 3. Επιβολή της σειράς που μου ζήτησες
# Προσοχή: Χρησιμοποιούμε τα νέα ονόματα με underscore (_)
sales_order = [
    "SalesOrderNumber",
    "OrderDate",
    "ProductKey",
    "ResellerKey",
    "EmployeeKey",
    "SalesTerritoryKey",
    "Quantity",
    "Unit_Price",
    "Sales",
    "Cost"
]

df_sales = df_sales.select(*sales_order)

# 4. Αποθήκευση στον Bronze πίνακα
df_sales.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze_layer.Sales_2017")

In [0]:
%sql
SELECT * FROM bronze_layer.Sales_2017

### **Validate**

In [0]:
%sql
DESCRIBE HISTORY bronze_layer.Sales_2017;

# **Sales(2018-2020)**

In [0]:
# 1. Διαβάζουμε το αρχείο
df_sales = spark.read.format("csv") \
    .option("sep", "\t") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
    .option("dateFormat", "yyyy-MM-dd") \
    .load("/Volumes/workspace/default/raw_data/Sales_2018.csv")

# 2. Αυτοματοποιημένος καθαρισμός ονομάτων (Unit Price -> Unit_Price)
for col_name in df_sales.columns:
    df_sales = df_sales.withColumnRenamed(col_name, col_name.replace(" ", "_").replace(";", "").replace("(", "").replace(")", ""))

# 3. Επιβολή της σειράς που μου ζήτησες
# Προσοχή: Χρησιμοποιούμε τα νέα ονόματα με underscore (_)
sales_order = [
    "SalesOrderNumber",
    "OrderDate",
    "ProductKey",
    "ResellerKey",
    "EmployeeKey",
    "SalesTerritoryKey",
    "Quantity",
    "Unit_Price",
    "Sales",
    "Cost"
]

df_sales = df_sales.select(*sales_order)

# 4. Αποθήκευση στον Bronze πίνακα
df_sales.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze_layer.Sales_2018")

In [0]:
%sql
SELECT * FROM bronze_layer.Sales_2018

### **Validate**

In [0]:
%sql
DESCRIBE HISTORY bronze_layer.Sales_2018;

# **Sales(2019-2020)**

In [0]:
# 1. Διαβάζουμε το αρχείο
df_sales = spark.read.format("csv") \
    .option("sep", "\t") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
    .option("dateFormat", "yyyy-MM-dd") \
    .load("/Volumes/workspace/default/raw_data/Sales_2019.csv")

# 2. Αυτοματοποιημένος καθαρισμός ονομάτων (Unit Price -> Unit_Price)
for col_name in df_sales.columns:
    df_sales = df_sales.withColumnRenamed(col_name, col_name.replace(" ", "_").replace(";", "").replace("(", "").replace(")", ""))

# 3. Επιβολή της σειράς που μου ζήτησες
# Προσοχή: Χρησιμοποιούμε τα νέα ονόματα με underscore (_)
sales_order = [
    "SalesOrderNumber",
    "OrderDate",
    "ProductKey",
    "ResellerKey",
    "EmployeeKey",
    "SalesTerritoryKey",
    "Quantity",
    "Unit_Price",
    "Sales",
    "Cost"
]

df_sales = df_sales.select(*sales_order)

# 4. Αποθήκευση στον Bronze πίνακα
df_sales.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze_layer.Sales_2019")

In [0]:
%sql
SELECT * FROM bronze_layer.Sales_2019

### **Validate**

In [0]:
%sql
DESCRIBE HISTORY bronze_layer.Sales_2018;

# **Sales(2020-2020)**

In [0]:
# 1. Διαβάζουμε το αρχείο
df_sales = spark.read.format("csv") \
    .option("sep", "\t") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
    .option("dateFormat", "yyyy-MM-dd") \
    .load("/Volumes/workspace/default/raw_data/Sales_2020.csv")

# 2. Αυτοματοποιημένος καθαρισμός ονομάτων (Unit Price -> Unit_Price)
for col_name in df_sales.columns:
    df_sales = df_sales.withColumnRenamed(col_name, col_name.replace(" ", "_").replace(";", "").replace("(", "").replace(")", ""))

# 3. Επιβολή της σειράς που μου ζήτησες
# Προσοχή: Χρησιμοποιούμε τα νέα ονόματα με underscore (_)
sales_order = [
    "SalesOrderNumber",
    "OrderDate",
    "ProductKey",
    "ResellerKey",
    "EmployeeKey",
    "SalesTerritoryKey",
    "Quantity",
    "Unit_Price",
    "Sales",
    "Cost"
]

df_sales = df_sales.select(*sales_order)

# 4. Αποθήκευση στον Bronze πίνακα
df_sales.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze_layer.Sales_2020")

In [0]:
%sql
SELECT * FROM bronze_layer.Sales_2020

### **Validate**

In [0]:
%sql
DESCRIBE HISTORY bronze_layer.Sales_2018;

drop table if exists bronze_layer.product;
drop table if exists bronze_layer.region;
drop table if exists bronze_layer.reseller;
drop table if exists bronze_layer.sales_2017;
drop table if exists bronze_layer.sales_2018;
drop table if exists bronze_layer.sales_2019;
drop table if exists bronze_layer.sales_2020;
drop table if exists bronze_layer.salesperson;
drop table if exists bronze_layer.salespersonregion;
drop table if exists bronze_layer.targets;

drop table if exists accenture_final_project.silver_layer.targets;
drop table if exists accenture_final_project.silver_layer.salesperson;

DROP CATALOG IF EXISTS accenture_final_project CASCADE;
